# PEGASE 9241 large-scale test

This notebook demonstrates the **full synthesis pipeline** on the large-scale PEGASE 9241 grid
(9 241 buses) — from loading the reference grid, extracting topological parameters,
generating a synthetic clone, and then assigning bus types, generation/load capacities,
dispatch, and transmission-line parameters.

In [ ]:
# ── Colab-only setup (run once, then restart the runtime) ──
# Colab pre-loads NumPy's C extensions into memory. Installing pandapower may
# upgrade NumPy, but the old .so files stay loaded and cause ImportError.
# Pinning versions + restarting the runtime fixes this.
%pip install -q "numpy>=2.2,<3" "scipy>=1.15,<2" "pandapower>=3.3,<4" "pypowsybl>=1.14,<2" "pypowsybl-jupyter>=1.0"
%pip install -q git+https://github.com/cookbook-ms/Chung_Lu_Chain-synthesizer.git@main

# Restart the Colab runtime so the new NumPy C extensions are loaded.
# After the restart, skip this cell and run the next one.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import pandapower as pp
import pandapower.networks as pn
import pandas as pd
import numpy as np
import networkx as nx

from powergrid_synth import (
    PowerGridGenerator,
    BusTypeAllocator,
    GraphComparator,
    CapacityAllocator,
    LoadAllocator,
    GenerationDispatcher,
    TransmissionLineAllocator,
    pandapower_to_nx,
    nx_to_pandapower,
    extract_topology_params_from_graph,
)

### Load the PEGASE 9241 grid using pandapower

In [ ]:
# 1. Load Real Grid and Convert
print("\n[1] Loading Reference Grid (PEGASE 9241)...")
net_real = pn.case9241pegase()
graph_real = pandapower_to_nx(net_real)
base_kv_list = graph_real.graph['base_kv_map']
print(f"Loaded {graph_real.number_of_nodes()} nodes and {graph_real.number_of_edges()} edges.")

## Generate a synthetic grid

### Extract Topology Characteristics from Graph

In [ ]:
print("\n[2] Analyzing Reference Topology...")
params = extract_topology_params_from_graph(graph_real)

### PowerGridGenerator

In [ ]:
print("\n[3] Generating Synthetic Clone...")
gen = PowerGridGenerator(seed=53)
synthetic_graph = gen.generate_grid(
    degrees_by_level=params['degrees_by_level'],
    diameters_by_level=params['diameters_by_level'],
    transformer_degrees=params['transformer_degrees'],
    keep_lcc=True
)

## Analysis

In [ ]:
# 5. Compare using the Library Module
print("\n[5] Running Comparative Analysis...")
comparator = GraphComparator(
    synth_graph=synthetic_graph, 
    ref_graph=graph_real, 
    synth_label="Synthetic", 
    ref_label="PEGASE 9241"
)

In [ ]:
comparator.plot_degree_comparison(log_scale=True, fig_size=(6,4), show_lines=True,)

In [ ]:
comparator.print_level_metrics()

In [ ]:
comparator.compare_degree_distributions()

Note that the large KS/RH values come from the ill-posed degree distributions on the corresponding voltage levels.

### Bus type assignment

In [ ]:
# 4. Allocate Bus Types
print("\n[4] Allocating Bus Types via AIS...")
allocator = BusTypeAllocator(synthetic_graph, entropy_model=0, bus_type_ratio=[80,60,0])
bus_types = allocator.allocate(max_iter=50)
nx.set_node_attributes(synthetic_graph, bus_types, name="bus_type")

In [ ]:
from collections import Counter

counts = Counter(bus_types.values())
total = sum(counts.values())
print(f"-----> Assignment Complete:")
print(f"       Generators: {counts['Gen']} ({counts['Gen']/total:.1%})")
print(f"       Loads:      {counts['Load']} ({counts['Load']/total:.1%})")
print(f"       Connectors: {counts['Conn']} ({counts['Conn']/total:.1%})")

### Generation capacities and load settings

In [ ]:
print("\n[6] Allocating Capacity...")
cap_allocator = CapacityAllocator(synthetic_graph)
capacities = cap_allocator.allocate()
total_cap = sum(capacities.values())
print(f"Total Generation: {total_cap:.2f} MW")
nx.set_node_attributes(synthetic_graph, capacities, name="pg_max")

In [ ]:
# Check top 10 generators
sorted_gens = sorted(capacities.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 Generators by Capacity:")
for node, cap in sorted_gens[:5]:
    print(f"  Node {node}: {cap:.2f} MW (Degree: {synthetic_graph.degree(node)})")

In [ ]:
print("\n[7] Allocating Loads ...")
load_allocator = LoadAllocator(synthetic_graph, ref_sys_id=1)
loads = load_allocator.allocate(loading_level='H')

nx.set_node_attributes(synthetic_graph, loads, name="pl")

total_load = sum(loads.values())
print(f"Total Load: {total_load:.2f} MW")
print(f"System Loading: {total_load/total_cap:.1%}")

In [ ]:
print("\n[8] Dispatching Generation...")
dispatcher = GenerationDispatcher(synthetic_graph, ref_sys_id=1)
dispatch = dispatcher.dispatch()
nx.set_node_attributes(synthetic_graph, dispatch, name="pg")
total_gen = sum(dispatch.values())
print(f"-> Total Power Dispatched: {total_gen:.2f} MW")
print(f"-> Generation Reserve: { total_cap - total_gen:.2f} MW")

In [ ]:
print("\n[9] Allocating Transmission Lines (Impedance & Capacity)...")
trans_allocator = TransmissionLineAllocator(synthetic_graph, ref_sys_id=1)
line_caps = trans_allocator.allocate()

total_lines = len(line_caps)
avg_cap = sum(line_caps.values()) / total_lines if total_lines > 0 else 0
print(f"-> Allocated {total_lines} Lines")
print(f"-> Average Line Capacity: {avg_cap:.2f} MVA")

### Convert to pandapower network

In [ ]:
synthetic_net = nx_to_pandapower(synthetic_graph, base_kv_map=base_kv_list)
synthetic_net

In [ ]:
pp.rundcpp(synthetic_net)
synthetic_net

## Compatible with pandapower

[pandapower](https://www.pandapower.org/) provides **Newton-Raphson AC** (`pp.runpp`) and **linear DC** (`pp.rundcpp`) power-flow solvers, and export to **JSON, Excel, SQLite, Pickle**.

> **Note:** `pp.runpp` (AC) may not converge for large synthetic grids; `pp.rundcpp` (DC) always works. For AC power flow on large grids, use pypowsybl's `run_ac` solver instead (shown below).

## Compatible with PowSyBl

[pypowsybl](https://pypowsybl.readthedocs.io/) provides **AC and DC load-flow solvers** (`run_ac`, `run_dc`), interactive grid visualisation, and export to **CGMES, XIIDM, MATPOWER, PSS/E, UCTE, AMPL, BIIDM, JIIDM**.

### Supported data export formats

PowerGridSynth provides a unified `GridExporter` class that wraps the built-in export functions of **pandapower** and **pypowsybl**, supporting **12+ industry-standard data formats** out of the box.

| Via | Formats | Methods |
|-----|---------|--------|
| **pandapower** | JSON, Excel, SQLite, Pickle | `to_json()`, `to_excel()`, `to_sqlite()`, `to_pickle()` |
| **pypowsybl** | CGMES, XIIDM, MATPOWER, PSS/E, UCTE, AMPL, BIIDM, JIIDM | `to_cgmes()`, `to_matpower()`, `to_psse()`, `to_pypowsybl(format=...)` |

In [ ]:
from powergrid_synth import GridExporter

exporter = GridExporter(synthetic_graph, base_kv_map=base_kv_list)

# --- pandapower-based exports ---
exporter.to_json("output/pegase9241_syn.json")

# --- pypowsybl-based exports ---
exporter.to_cgmes("output/pegase9241_syn_cgmes")
exporter.to_matpower("output/pegase9241_syn")
exporter.to_psse("output/pegase9241_syn")
exporter.to_pypowsybl("output/pegase9241_syn.xiidm", format="XIIDM")

In [ ]:
import pypowsybl as ppl

#### Convert from pandapower dataframe to pypowsybl network

In [ ]:
from powergrid_synth import pandapower_to_pypowsybl

In [ ]:
ppl_net = pandapower_to_pypowsybl(synthetic_net)

In [ ]:
ppl_net

#### Run AC PF solver using PowSyBl

In [ ]:
ppl.loadflow.run_ac(ppl_net)